# Этап D: дообучение ретривераОдна тетрадь на весь рецепт. Шаги независимы: каждый проверяет, нетли готового результата, и пропускает себя, — поэтому после обрыва сессиидостаточно запустить ячейки заново.Перед запуском:* **+ Add Input** → датасет `ru-fin-retrieval` (код и корпус), датасет  с набором запросов (`queries.jsonl`, `split.json`, `synthetic_train.jsonl`,  `synthetic_dev.jsonl`, `qrels.tsv`, `podvyborki.json`) **и** датасет  с канонической нарезкой (`chunks_base.jsonl.gz`, он же уехал с выдачами  этапа B);* **Settings** → **Accelerator** → **GPU T4**, интернет включён;* **токен HuggingFace подключён к этой тетради.** Секреты Kaggle  привязываются к конкретной тетради через браузер и новым ядром  не наследуются. Один раз проект на этом потерял два часа квоты.**Что здесь нельзя сделать случайно.** Тест трогается один раз, на шаге`final`. Пока подбирается рецепт, `run_phase_d.py --step train` считаеттолько по dev и без dev отказывается работать вовсе. Смотреть на тестмежду этапами значит подгонять под него выбор рецепта, и тогда итоговаяцифра перестаёт что-либо значить.Порядок и цена:| шаг | что делает | оценка ||---|---|---:|| 4 | проба на урезанной выборке | 15 мин || 5 | **D0** трудные негативы и оценки учителя | 1–1,5 ч || 6 | нулевая точка: dev необученного ученика | 10 мин || 7 | **D1–D5** рецепт: обучение и dev после каждого этапа | 3–4 ч || 8 | **D6** финал: один прогон по всему набору | 20 мин || 9 | забывание на посторонних данных | 20 мин |

## 1. Карта и токенДолжна быть видна Tesla T4. Токен проверяется до всякого долгого прогона.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

from kaggle_secrets import UserSecretsClient
try:
    token = UserSecretsClient().get_secret("HF_TOKEN")
    import os
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    print("токен HuggingFace на месте, длина", len(token))
except Exception as e:
    print("ТОКЕНА НЕТ:", e)
    print("Add-ons -> Secrets -> HF_TOKEN. Без него скачивание весов "
          "режется по скорости и может встать намертво")

## 2. БиблиотекиvLLM здесь не нужен вовсе: языковая модель на этом этапе не запускается,и установка занимает минуту, а не пятнадцать.Нужны `sentence-transformers` не ниже 3.0 (в более ранних нет`SentenceTransformerTrainer`, на котором держится обучение двумя наборамисразу), `datasets`, `accelerate` и `mteb` для замера забывания.

In [ ]:
import subprocess, sys, time

t = time.time()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U',
                'sentence-transformers>=3.0', 'datasets', 'accelerate', 'mteb',
                '--progress-bar', 'off'], check=True)
print(f'установка заняла {(time.time() - t) / 60:.1f} мин')

import sentence_transformers, torch, transformers
print('sentence-transformers', sentence_transformers.__version__)
print('transformers', transformers.__version__, '| torch', torch.__version__)
print('карта:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ')

# На T4 (Turing) bf16 нет вовсе. Обучение идёт в fp32 — это решение рецепта,
# и в конфигурации нет поля, которым половинную точность можно включить.
print('bf16 поддерживается:', torch.cuda.is_bf16_supported()
      if torch.cuda.is_available() else '—')
from sentence_transformers.losses import (CachedMultipleNegativesRankingLoss,
                                          MatryoshkaLoss)
print('нужные функции потерь на месте')

## 3. Код и данные на место

In [ ]:
import glob, os, shutil, zipfile

# Kaggle монтирует датасеты по-разному: то прямо в /kaggle/input/<имя>,
# то глубже, внутри /kaggle/input/datasets/<владелец>/<имя>. Поэтому путь
# не задаётся, а источник ищется по содержимому: нужен каталог, где лежит
# пакет rufin.
def find_source(root='/kaggle/input', max_depth=4):
    found, archives = [], []
    for cur, dirs, files in os.walk(root):
        if cur.count('/') - root.count('/') >= max_depth:
            dirs.clear()
        if 'rufin' in dirs:
            stale = ('__notebook__.ipynb' in files or 'embeddings' in dirs
                     or '__output__.json' in files)
            found.append((1 if stale else 0, cur))
        archives += [os.path.join(cur, f) for f in files if f.endswith('.zip')]
    if found:
        found.sort()
        return found[0][1], None
    return (os.path.dirname(archives[0]), archives[0]) if archives else (None, None)


src, archive = find_source()
if src is None:
    for cur, dirs, files in os.walk('/kaggle/input'):
        depth = cur.count('/') - '/kaggle/input'.count('/')
        if depth >= 3:
            dirs.clear()
        print('   ' * depth + os.path.basename(cur) + '/')
        for f in sorted(files)[:6]:
            print('   ' * (depth + 1) + f)
    raise SystemExit('датасет не подключён: панель справа -> + Add Input')

os.makedirs('/kaggle/working', exist_ok=True)
if archive:
    with zipfile.ZipFile(archive) as z:
        z.extractall('/kaggle/working')
else:
    for имя in os.listdir(src):
        путь = os.path.join(src, имя)
        цель = os.path.join('/kaggle/working', имя)
        if os.path.isdir(путь):
            shutil.copytree(путь, цель, dirs_exist_ok=True)
        elif имя.endswith(('.py', '.md', '.gz')):
            shutil.copyfile(путь, цель)
print('код на месте:', sorted(f for f in os.listdir('/kaggle/working')
                              if f.endswith('.py'))[:8], '...')

# Обучению нужны именно эти файлы, и каждый — по своей причине.
for имя, зачем in (('split.json', 'кандидаты только из обучающих актов'),
                   ('synthetic_train.jsonl', 'сами обучающие пары'),
                   ('queries.jsonl', 'тексты dev-запросов'),
                   ('qrels.tsv', 'эталоны dev'),
                   ('podvyborki.json', 'состав dev'),
                   ('chunks_base.jsonl', 'каноническая нарезка')):
    нашлось = (glob.glob(f'/kaggle/input/**/{имя}', recursive=True)
               + glob.glob(f'/kaggle/input/**/{имя}.gz', recursive=True)
               + glob.glob(f'/kaggle/working/{имя}'))
    print(f'{имя:<24} {"есть" if нашлось else "НЕТ":<5} {зачем}')

### 3б. Каноническая нарезка на место. ОбязательноНарезка детерминирована, но зависит от версии токенизатора. Идентификаторфрагмента позиционный (`акт#номер`), поэтому при сдвиге границ онне исчезает, а начинает указывать на другой текст: эталон обучающейвыборки съехал бы молча, и модель училась бы находить не то.Поэтому нарезка не пересчитывается, а берётся готовая — та же, по которойсчитались выдачи, разметка и обучающие вопросы. Ячейка должна напечатать**62594**. Кэш сносится первой строкой: упавший прогон успевает записатьтуда свою нарезку, а готовая берётся раньше, чем считается новая.

In [ ]:
import glob, gzip, os, shutil

dst = '/kaggle/working/chunks/base.jsonl'
if os.path.exists(dst):
    было = sum(1 for _ in open(dst, encoding='utf-8'))
    print(f'сношу кэш: {было} фрагментов')
    os.remove(dst)

найдено = (glob.glob('/kaggle/input/**/chunks_base.jsonl', recursive=True)
           + glob.glob('/kaggle/input/**/chunks_base.jsonl.gz', recursive=True))
assert найдено, 'нет chunks_base.jsonl: подключите датасет с нарезкой'
src = найдено[0]

os.makedirs('/kaggle/working/chunks', exist_ok=True)
opener = gzip.open if src.endswith('.gz') else open
with opener(src, 'rb') as fi, open(dst, 'wb') as fo:
    shutil.copyfileobj(fi, fo, length=1 << 20)

n = sum(1 for _ in open(dst, encoding='utf-8'))
print('источник:', src)
print('фрагментов:', n)
assert n == 62594, f'ожидалось 62594, а не {n}: нарезка не каноническая'

## 4. Проба. Пятнадцать минут вместо пяти часовПодготовка на пятидесяти вопросах и обучение на двухсот парах. Проверяетсяне качество — его на таких числах нет, — а то, что работает весь путь:кандидаты, оценки учителя, отсечка негативов, заморозка словаря, обучение,оценка по dev и запись в журнал.Журнал и папка весов **отдельные**: проба не должна попасть в таблицурецепта как результат.

In [ ]:
!cd /kaggle/working && python gpu_prepare.py --limit 50 --depth 10 \
    --out /kaggle/working/proba

import json
rec = [json.loads(l) for l in open(
    '/kaggle/working/proba/train_prepared.jsonl', encoding='utf-8')]
print(f'\nзаписей {len(rec)}, у первой кандидатов {len(rec[0]["candidates"])}')
print('место эталона у учителя:', [r['teacher_place'] for r in rec[:10]])
print('оценка эталона:', round(rec[0]['gold_score'], 2),
      '| лучшая у кандидата:', round(rec[0]['candidates'][0][1], 2))
print(json.dumps(json.load(open('/kaggle/working/proba/report_prepare.json',
                                encoding='utf-8'))['негативы'],
                 ensure_ascii=False, indent=1)[:900])

In [ ]:
# Нулевая точка пробы и один этап на двухсот парах. Оценка по dev считается
# целиком — это 62 594 фрагмента, около четырёх минут, и урезать её нельзя:
# она и есть то, что проверяется.
!cd /kaggle/working && python run_phase_d.py --step baseline \
    --journal /kaggle/working/proba/journal.json

!cd /kaggle/working && python run_phase_d.py --step train --tags a-1ep \
    --limit-train 200 \
    --prepared /kaggle/working/proba/train_prepared.jsonl \
    --weights /kaggle/working/proba/weights \
    --journal /kaggle/working/proba/journal.json

## 5. D0: трудные негативы и оценки учителяСамая дорогая часть этапа и единственная, которая считается один разна все остальные: негативы берёт отсюда этап B, оценки для дистилляции —этап C.Глубина 30, как у третьего фильтра этапа C: это 183 тысячи проходовкросс-энкодера. Увеличивать её значит увеличивать время линейно, а списокдля дистилляции всё равно берётся из первых восьми.Файл маленький (единицы мегабайт) и **его надо сохранить отдельно**:если сессия оборвётся после D0, повторять полтора часа незачем — достаточноподключить его входом следующему прогону.

In [ ]:
!cd /kaggle/working && python gpu_prepare.py --depth 30 \
    --out /kaggle/working/train

import json
r = json.load(open('/kaggle/working/train/report_prepare.json', encoding='utf-8'))
for k, v in r.items():
    if isinstance(v, dict):
        print(f'{k}:')
        for k2, v2 in v.items():
            print(f'   {k2:<34} {v2}')
    else:
        print(f'{k:<34} {v}')

## 6. Нулевая точка: dev необученного ученикаЦифра есть и в отчёте этапа 2 (dev NDCG@10 0,449), но взять её оттуданельзя: сравнение этапов парное, и для него нужны поквериные значения,посчитанные этим же кодом на этом же корпусе. Десять минут видеокартыдешевле, чем сомнение в главной цифре отчёта.

In [ ]:
!cd /kaggle/working && python run_phase_d.py --step baseline

## 7. D1–D5: рецептЭтапы считаются по порядку, и после каждого — dev и вердикт. Сеткаэтапа E строится от состава лучшего принятого этапа, поэтому её имеетсмысл запускать вторым вызовом, когда предыдущие уже в журнале.Каждый этап обучается **с исходных весов**: рецепт наращивается составом,а не дообучением по кругу. Иначе «две эпохи» перестаёт означать две эпохи,а отклонённый этап всё равно оставляет след в весах.Если прогон оборвётся, запустите ячейку снова: этапы, уже записанныев журнал, пропускаются. Переделать этап нарочно — ключ `--force`;запас квоты позволяет переделать любой трижды.

In [ ]:
# Этапы A, A+, B, C, D. Дальше — сетка E от лучшего принятого.
!cd /kaggle/working && python run_phase_d.py --step train

In [ ]:
# Второй вызов: сетка этапа E уже знает, какой состав оказался лучшим.
!cd /kaggle/working && python run_phase_d.py --step train

## 8. D6: финал. Единственное касание тестаОдин прогон по всему набору запросов. Метрики по заголовочной подвыборкеи по живым вопросам отдельно считает **локальный** `make metrics` по этимже файлам выдач и тем же кодом, которым посчитаны все остальныеконфигурации. Разбивка важнее итоговой цифры: если прирост естьна заголовочной и нет на живых, модель выучила язык генератораобучающих вопросов, а не домен, и в отчёт идёт именно это.Заодно сохраняется матрица эмбеддингов корпуса: по ней на маке снимаетсязадержка и размер индекса, а размерности матрёшки получаются из неё срезом.

In [ ]:
!cd /kaggle/working && python run_phase_d.py --step final

## 9. Забывание: что испортилось снаружи доменаДообучение на узком домене портит общее качество — это свойство метода,и единственный способ что-то о нём сказать — померить числом на наборе,которого в обучении не было вовсе.Мерится и лучший этап, и этап A: если дистилляция удерживает общеекачество лучше, чем чистый контрастив, это ещё один довод за неё,и довод измеренный.Шаг необязательный и падать не умеет: если `mteb` не поставился илисменил устройство объекта задачи, он напечатает это и вернёт управление.

In [ ]:
!cd /kaggle/working && python run_phase_d.py --step forget --tags a-1ep

## 10. Что получилось

In [ ]:
import json, os, sys
sys.path.insert(0, '/kaggle/working')
from rufin.training.journal import Journal

j = Journal.load('/kaggle/working/train/journal.json')
print(j.table())
лучшее = j.best()
базовая = j.baseline()
if лучшее and базовая:
    print(f'\nлучшее принятое: {лучшее.tag}, dev NDCG@10 {лучшее.ndcg:.3f} '
          f'против {базовая.ndcg:.3f} у необученного ученика '
          f'({лучшее.ndcg - базовая.ndcg:+.3f})')
for имя in ('runs/report_phase_d.json', 'runs/zabyvanie.json'):
    путь = os.path.join('/kaggle/working', имя)
    if os.path.exists(путь):
        print(f'\n--- {имя}')
        print(json.dumps(json.load(open(путь, encoding='utf-8')),
                         ensure_ascii=False, indent=1)[:1500])

## 11. Скачать. **До** всякого Save VersionВ интерактивной сессии вывод не публикуется сам: `/kaggle/working`живёт, пока жива сессия. **Save & Run All (Commit)** его не спасает,а уничтожает — он чистит каталог и считает всё заново. **Quick Save**без **Advanced settings** файлы тоже не сохраняет.Что куда класть на маке:* `runs/base__dense-rufin*.jsonl` → `data/runs/`;* `train/journal.json`, `report_prepare.json`, `runs/report_phase_d.json`,  `runs/zabyvanie.json` → `docs/raw/`;* `train/train_prepared.jsonl` → `data/queries/` (по нему `make check-train`  проверяет отсечку негативов);* `embeddings/base/rufin/` целиком → `data/embeddings/base/rufin/`  (48 МБ, нужна для `make latency` и `make matryoshka`);* веса лучшего этапа — отдельной выгрузкой, они полгигабайта.Дальше на маке: `make metrics`, `make recipe`, `make matryoshka`,`make latency DENSE=rufin`.

In [ ]:
import os, shutil
from IPython.display import FileLink

# Мелкое едет одним архивом: вывозить по файлу из панели Output дольше
# и легко забыть половину. Веса и матрица эмбеддингов — отдельно,
# они крупные.
os.makedirs('/kaggle/working/vyvoz', exist_ok=True)
взято = []
for корень, образцы in (('/kaggle/working/runs', ('.jsonl', '.json')),
                        ('/kaggle/working/train', ('.json', '.jsonl'))):
    for база, _, файлы in os.walk(корень):
        for имя in файлы:
            if имя.endswith(образцы):
                путь = os.path.join(база, имя)
                if os.path.getsize(путь) > 200 * 1024 * 1024:
                    continue
                shutil.copy2(путь, os.path.join('/kaggle/working/vyvoz', имя))
                взято.append((имя, os.path.getsize(путь) // 1024))

for имя, кб in sorted(set(взято)):
    print(f'   {имя:<34} {кб:>8} КБ')
assert взято, 'нечего вывозить: ни один шаг не давал файлов'
shutil.make_archive('/kaggle/working/vyvoz_etap_d', 'zip', '/kaggle/working/vyvoz')
print('\nархив:', os.path.getsize('/kaggle/working/vyvoz_etap_d.zip') // 1024, 'КБ')

# матрица эмбеддингов: 48 МБ, нужна на маке для задержки и матрёшки
эмб = '/kaggle/working/embeddings/base/rufin'
if os.path.exists(эмб):
    shutil.make_archive('/kaggle/working/vyvoz_embeddings', 'zip', эмб)
    print('матрица:', os.path.getsize('/kaggle/working/vyvoz_embeddings.zip')
          // 1048576, 'МБ')
FileLink('vyvoz_etap_d.zip')